# Multi-label ViT notebook

In this notebook we implement a vision transformer (Dinov3), pretrained on satellite imagery. We finetune it on the multi-label training dataset, and then test the performance on the validation dataset.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
# Internal import
import deep_learning_land_use_classification.config as config
import deep_learning_land_use_classification.evaluation as evaluation
from deep_learning_land_use_classification.dataset import get_multi_label_data
import deep_learning_land_use_classification.wanddb_helpers as wh
from deep_learning_land_use_classification.early_stopping import EarlyStopper

# External imports
import torch
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import torch.nn as nn
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import re


c:\Users\tomer\anaconda3\envs\Deep-learning-land-use-classification\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Getting data
The data is split into training (64%), validation (16%) and test (20%). Training is used for model training, validation is used to tune hyperparameters and pick the best model, and testing is used to assess the performance of the final model. Data is split using multi-label stratified sampling from the iterative-stratification library.

## Getting data
The data is split into training (64%), validation (16%) and test (20%). Training is used for model training, validation is used to tune hyperparameters and pick the best model, and testing is used to assess the performance of the final model. Data is split using stratified sampling.

In [2]:
train_df, test_df, val_df,class_names, num_classes = get_multi_label_data()

In [3]:
# Wandb initialization for experiment tracking
run = wh.init_run(
    run_name="ViT unfrozen augmentation no color 02",
    task="multi",
    architecture="dinov3-vitl16",
    num_classes=num_classes,
    loss="BCEWithLogitsLoss",
    epochs=50,
    batch_size=32,
    learning_rate=1e-3,
    optimizer="AdamW",
    pretrained=True,
    pretraining_dataset="sat493m",
    pretraining_source="huggingface",
    weights="facebook/dinov3-vitl16-pretrain-sat493m",
    model_name=None,
    augmentation=True,
    early_stopping=True,
    patience=6,
    min_delta=0.001,
    backbone_frozen=False,
    backbone_learning_rate=1e-4,
    dropout=0.3,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\tomer\_netrc.
wandb: Currently logged in as: tomer-peled (sstaheli52-wageningen-university-and-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Define model

In [ ]:
# Define model and processor
pretrained_model_name = "facebook/dinov3-vitl16-pretrain-sat493m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)

In [ ]:
# Define data augmentation for training
train_augs = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    # transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    transforms.RandomRotation(degrees=90)
])

In [ ]:
class MultiLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None, augment=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")
        
        if self.augment: # Apply augmentations only during training
                image = self.augment(image)
                
        inputs = processor(images=image, return_tensors="pt") # converts image to tensor and normalizes it
        inputs = {k: v.squeeze(0) for k, v in inputs.items()} # remove batch dimension

        label = torch.tensor(row[self.class_names].values.astype(np.float32))
            
        return inputs, label

In [7]:
# Create datasets and dataloaders
train_dataset = MultiLabelDataset(train_df, class_names, processor, augment=train_augs)
val_dataset  = MultiLabelDataset(val_df, class_names, processor, augment=None)

train_loader = DataLoader(train_dataset, batch_size=run.config.batch_size, shuffle=True)
val_loader  = DataLoader(val_dataset, batch_size=run.config.batch_size, shuffle=False)

In [ ]:
# this is a simple wrapper around the DINO backbone to add a classification head on top
class DinoClassifier(torch.nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.0):
        super().__init__()
        self.backbone = backbone
        self.dropout = torch.nn.Dropout(p=dropout)
        self.classifier = torch.nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values) # get the output of the backbone
        features = outputs.pooler_output
        features = self.dropout(features)
        return self.classifier(features)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

backbone = AutoModel.from_pretrained(pretrained_model_name)

model = DinoClassifier(backbone, num_classes=num_classes, dropout=run.config.dropout).to(device)
model = model.to(device)

Using device: cuda


Loading weights: 100%|██████████| 415/415 [00:00<00:00, 2597.33it/s]


In [ ]:
labels = train_df[class_names].values

# Count positives per class
pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

# Avoid division by zero
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32)

# Implement BCEWithLogitsLoss with pos_weight to handle class imbalance
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

# Freeze all backbone layers to prevent them from being updated during training, to save memory and speed up training.
for p in model.backbone.parameters():
    p.requires_grad = False

# Unfreeze only the last transformer block for light fine-tuning. Improves performance.
for p in model.backbone.model.layer[-1].parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {"params": model.classifier.parameters(), "lr": run.config.learning_rate},
        {"params": model.backbone.model.layer[-1].parameters(), "lr": run.config.backbone_learning_rate},
    ],
    weight_decay=0.01
)

In [ ]:
# Log to wandb
wh.log_model_summary(run, model)

### Train Model

In [12]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for inputs, labels in loader:
        pixel_values = inputs["pixel_values"].to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [13]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, labels in loader:
            pixel_values = inputs["pixel_values"].to(device)
            labels = labels.to(device)

            outputs = model(pixel_values)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [15]:
epochs = run.config.epochs
early_stopper = EarlyStopper(patience=run.config.patience, min_delta=run.config.min_delta)

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss  = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"val Loss:  {val_loss:.4f}")
    precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics_multilabel(
        model,
        val_loader,
        device,
        threshold=0.5    
    )
    
    wh.log_epoch(run, epoch, train_loss, val_loss,
             precision, recall, f1, p_macro, r_macro, f1_macro, class_names)
    
    # Early stopping check
    if early_stopper.step(val_loss, model):
        print(f"Early stopping triggered at epoch {epoch+1}. Best val loss: {early_stopper.best_loss:.4f}")
        break

# Restore the weights from the best epoch
early_stopper.restore_best_weights(model)
print("Restored best model weights.")
    
run.finish()

Epoch 1/50
Train Loss: 0.7929
val Loss:  0.6729
Epoch 2/50
Train Loss: 0.4357
val Loss:  0.5527
Epoch 3/50
Train Loss: 0.3120
val Loss:  0.4645
Epoch 4/50
Train Loss: 0.2499
val Loss:  0.3642
Epoch 5/50
Train Loss: 0.2003
val Loss:  0.3255
Epoch 6/50
Train Loss: 0.1815
val Loss:  0.2764
Epoch 7/50
Train Loss: 0.1530
val Loss:  0.2631
Epoch 8/50
Train Loss: 0.1378
val Loss:  0.2416
Epoch 9/50
Train Loss: 0.1229
val Loss:  0.2270
Epoch 10/50
Train Loss: 0.1160
val Loss:  0.2146
Epoch 11/50
Train Loss: 0.1079
val Loss:  0.2209
Epoch 12/50
Train Loss: 0.1012
val Loss:  0.1991
Epoch 13/50
Train Loss: 0.0923
val Loss:  0.1978
Epoch 14/50
Train Loss: 0.0865
val Loss:  0.2092
Epoch 15/50
Train Loss: 0.0847
val Loss:  0.1940
Epoch 16/50
Train Loss: 0.0769
val Loss:  0.1860
Epoch 17/50
Train Loss: 0.0716
val Loss:  0.1866
Epoch 18/50
Train Loss: 0.0676
val Loss:  0.1887
Epoch 19/50
Train Loss: 0.0627
val Loss:  0.1921
Epoch 20/50
Train Loss: 0.0620
val Loss:  0.1770
Epoch 21/50
Train Loss: 0.057

class/airplane/f1,▁▁▃▅▃▇▇████▇▇█▇████████████████████
class/airplane/precision,▁▁▃▄▂▆▆████▆▇█▇████████████████████
class/airplane/recall,███████████████████████▁▁▁███████▁█
class/bare_soil/f1,▁▂▄▄▄▄▆▆▅▆▇▇█▇███▇▇██▇███▇███▇██▆▇█
class/bare_soil/precision,▁▂▆▅▆▄▇▆▇▆▆▇▇▆▇██▅▆█▇▅▆▇▇▅█▇▇███▄▇▇
class/bare_soil/recall,▃▃▁▁▂▅▃▄▂▅▆▅▇▅▆▅▅▇▆▅▆█▇▆▆▇▅▆▆▅▅▅█▆▆
class/buildings/f1,▁▃▅▅▄▆▇▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇█████▇
class/buildings/precision,▁▂▄▅▄▆▇▆▆▆▇▇▇▇▇▇▇█▇▇▇▇█▇▇▇█▇███▇██▇
class/buildings/recall,▇█▅▃▆▂▂▄▅▆▃▂▃▂▃▃▂▂▄▄▅▂▁▄▆▁▂▃▂▅▃▆▃▂▂
class/cars/f1,▁▂▄▅▅▄▄▄▄▄▄▄▅▅▄▅▆▅▅▅▅▇▅▇▇▇▅▆▆▅▇▆▆▆█
+47,...


## Threshold sweep

Here we check how changing the threshold value changes model performance by comparing threshold values ranging from 0.05 to 0.95 and their resulting F1-score.

In [ ]:
thresholds = np.arange(0.05, 1.0, 0.05)
f1_scores = []

for threshold in thresholds:
    _, _, _, _, _, f1_macro = evaluation.compute_accuracy_metrics_multilabel(
        model,
        val_loader,
        device,
        threshold=threshold    
    )
    f1_scores.append(f1_macro)
    print(f"Threshold: {threshold:.2f} | F1-Macro: {f1_macro:.4f}")

# Find best threshold
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"\n✓ Best Threshold: {best_threshold:.2f} with F1-Score: {best_f1:.4f}")

# Create figure
plt.figure(figsize=(10, 6))
plt.plot(thresholds, f1_scores, marker='o', linewidth=2, markersize=6, label='F1-Score')
plt.scatter([best_threshold], [best_f1], color='red', s=200, marker='*', 
           label=f'Best: {best_threshold:.2f} (F1={best_f1:.4f})', zorder=5)

plt.xlabel('Threshold', fontsize=12)
plt.ylabel('F1-Score (Macro)', fontsize=12)
plt.title('Threshold Sweep - F1-Score vs Classification Threshold', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(thresholds, [f'{t:.2f}' for t in thresholds], rotation=45)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Log to wandb
wh.run.log({"threshold_sweep": plt})

## Save model for later use

In [16]:
# Save a full training checkpoint (weights + key metadata) for reproducibility.
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)

safe_run_name = re.sub(r"[^a-zA-Z0-9_-]+", "-", run.name.strip()).strip("-").lower()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_name = f"{safe_run_name}_dino_multilabel_{timestamp}.pt"
checkpoint_path = config.MODELS_DIR / checkpoint_name

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "model_class": model.__class__.__name__,
    "backbone_name": pretrained_model_name,
    "num_classes": num_classes,
    "class_names": list(class_names),
    "run_name": run.name,
    "run_id": run.id,
    "epoch": epoch + 1 if "epoch" in locals() else None,
    "dropout": float(run.config.dropout) if "dropout" in run.config else None,
    "learning_rate": float(run.config.learning_rate) if "learning_rate" in run.config else None,
    "backbone_learning_rate": float(run.config.backbone_learning_rate) if "backbone_learning_rate" in run.config else None,
    "batch_size": int(run.config.batch_size) if "batch_size" in run.config else None,
}

torch.save(checkpoint, checkpoint_path)
print(f"Saved checkpoint to: {checkpoint_path}")

NameError: name 're' is not defined